# Person 1 — Dataset Investigation and EDA

Phase 3 investigates class balance, duration, frame energy, candidate MFCCs, and fixed-length windows for the Swahili audio clips. It does not train a classifier and it does not rewrite `data/splits/shared_split.csv`.

Phase 4 adds a configurable feature pipeline and checks it on a small subset of the shared training split. The defaults are not a selected preprocessing configuration. This notebook still does not train a classifier.

The official `Test.csv` file has no labels and is not the research test set.


## 1. Setup

The shared split uses seed 42. This notebook calls `run_phase3`, which reads that file and checks it in memory. Re-running the cell repeats the labeled-audio pass. It does not save a new split.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Image, display
import pandas as pd

from src.data.eda import figures_dir, results_eda_dir
from src.data.phase3 import run_phase3
from src.utils.reproducibility import SHARED_SPLIT_SEED, set_seed

set_seed(SHARED_SPLIT_SEED)
payload = run_phase3(ROOT)
print("seed", SHARED_SPLIT_SEED)
print("split rewritten", False)
print("saved split matches seed 42", payload["split"]["matches_recomputed_seed_42_split"])
print("split counts", payload["split"]["split_counts"])


## 2. Load metadata

Phase 2 stored one row per labeled clip. The next cell shows only the first rows and the column names, not the full table.


In [ ]:
metadata_path = results_eda_dir(ROOT) / "audio_metadata_train.csv"
metadata = pd.read_csv(metadata_path)
print(metadata.shape)
print(list(metadata.columns))
metadata.head()


## 3. Class balance

Equal label counts do not justify class weighting or oversampling. Validation and internal-test counts of 52 or 53 are the integer result of taking 15 percent of 350.


In [ ]:
balance = payload["class_balance"]
print(balance["examples_per_class"])
print(balance["split_ranges"])
print(balance["reason"])
display(pd.read_csv(results_eda_dir(ROOT) / "class_balance_by_split.csv"))
display(Image(filename=str(figures_dir(ROOT) / "class_distribution.png")))


## 4. Duration analysis

Thresholds such as 10 seconds are descriptive counts. No clip is removed, and no maximum duration is chosen here.


In [ ]:
print(payload["duration"]["scopes"]["labeled_pool"]["summary"])
display(pd.read_csv(results_eda_dir(ROOT) / "duration_thresholds.csv"))
display(pd.read_csv(results_eda_dir(ROOT) / "duration_by_class_detailed.csv"))
display(Image(filename=str(figures_dir(ROOT) / "audio_duration_distribution.png")))
display(Image(filename=str(figures_dir(ROOT) / "duration_distribution_log.png")))
display(Image(filename=str(figures_dir(ROOT) / "duration_mean_median_by_class.png")))


## 5. Temporal and energy analysis

A frame is low-energy when its RMS is below 1 percent of that clip's maximum frame RMS. That proportion is not a silence percentage. The Phase 2 sample-level column is the fraction of samples below 1 percent of the clip peak, which is a different rule.


In [ ]:
temporal = payload["temporal"]
print(temporal["rule"]["definition"])
print(temporal["labeled_pool"])
display(pd.DataFrame(temporal["by_duration_bin"]))
display(Image(filename=str(figures_dir(ROOT) / "low_energy_edges_by_duration.png")))
display(Image(filename=str(figures_dir(ROOT) / "representative_waveforms.png")))
display(Image(filename=str(figures_dir(ROOT) / "representative_spectrograms.png")))


## 6. Spectral and MFCC investigation

The candidate settings are a 25 ms Hann window, a 10 ms hop, 40 HTK-mel filters, and 13, 20, or 40 DCT coefficients. None of these settings is adopted in this phase. Delta features were checked for feasibility only.


In [ ]:
mfcc = payload["mfcc"]
print("shape mismatches", mfcc["mfcc_shape_mismatches"])
print("delta check", mfcc["delta_check"])
display(pd.read_csv(results_eda_dir(ROOT) / "mfcc_candidates.csv"))
display(Image(filename=str(figures_dir(ROOT) / "mfcc_examples.png")))


## 7. Fixed-length investigation

Window statistics that could guide a later choice are reported on the shared training split. The internal test split is not used to pick a window. No window is selected here.


In [ ]:
windows = pd.read_csv(results_eda_dir(ROOT) / "fixed_window_impact.csv")
display(windows.loc[windows["scope"] == "shared_train_split"])
display(Image(filename=str(figures_dir(ROOT) / "fixed_window_impact.png")))


## 8. Data quality checks

Exact PCM SHA-256 was computed for every labeled clip. Pairwise near-duplicate comparison was not run.


In [ ]:
quality = payload["quality"]
print("sample rates", quality["sample_rate_counts"])
print("channels", quality["channel_counts"])
print("zero length", quality["zero_length_clips"])
print("full-scale peaks", quality["n_full_scale_peak_at_least_32767"])
print("pcm duplicate groups", quality["n_duplicate_pcm_hash_groups"])
print(quality["pairwise_audio_similarity"])
display(pd.read_csv(results_eda_dir(ROOT) / "representative_examples.csv"))


## 9. Summary of findings

Decisions that are supported are limited to format consistency and the absence of label imbalance. Duration handling, trimming, MFCC size, deltas, and aggregation were not selected in this investigation. Section 10 checks the pipeline that exposes those options.


In [ ]:
display(pd.read_csv(results_eda_dir(ROOT) / "preprocessing_investigation.csv"))
print((results_eda_dir(ROOT) / "eda_findings.md").read_text(encoding="utf-8"))


## 10. Preprocessing pipeline validation

The pipeline exposes duration, MFCC, delta, normalization, and aggregation settings. The checks below use a small deterministic subset of the shared training split. They do not train a classifier, do not read the internal test split, and do not read the official `Test.csv`.

Six seconds and eight seconds are both run. Neither duration is a selected window. Feature values are not printed.

In [ ]:
from IPython.display import display

from dataclasses import replace
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import CONFIGURATION_STATUS, PreprocessingConfig, feature_dimension
from src.preprocessing.validate import run_phase4_validation

print(CONFIGURATION_STATUS)
static_mean_std = PreprocessingConfig(duration_seconds=6.0, aggregation="mean_std")
eight_seconds = replace(static_mean_std, duration_seconds=8.0)
deltas = replace(static_mean_std, include_delta=True, include_delta_delta=True)
print("6 s, 13 MFCC, mean+std", feature_dimension(static_mean_std))
print("8 s, 13 MFCC, mean+std", feature_dimension(eight_seconds))
print("6 s, 13 MFCC + delta + delta-delta, mean+std", feature_dimension(deltas))

report = run_phase4_validation(ROOT)
columns = ["check", "feature_dimension", "all_finite", "reproducible", "passed"]
display(report["table"].loc[:, columns])
print("all checks passed", report["all_passed"])
print("training clips", report["n_examples"], "classes", report["n_classes"])
print("source audio modified", report["source_audio_modified"])
print("shared split modified", report["shared_split_modified"])
